# 40 Align EEG To BOLD TRs And Build Keep Masks

## What This Notebook Does

This is the public Stage-4 alignment notebook. It preserves the original alignment logic that:
- reconciles the raw and preprocessed EEG timelines,
- maps merged EEG exclusion intervals back onto the raw time axis,
- projects usable EEG coverage onto the BOLD TR grid,
- applies the preserved hybrid keep-mask rules and EEG sample-completeness gate,
- writes same-TR EEG parcel power for the final no-lag design,
- and also writes the lagged EEG feature branch needed for optional provenance analyses.

## When To Run It

Run this notebook before `41_build_final_no_lag_fusion_observation_segments.ipynb`.

This notebook sits downstream of:
- the cleaned Stage-1 exclusion-union exports
- the cleaned Stage-2 EEG parcel exports
- the cleaned Stage-3 BOLD parcel exports

## Manuscript Linkage

- Main Methods 2.4
- Supplementary Methods 1.5
- Figure 1 support
- Table S6 support
- Table S7 support

## Inputs Expected

- raw EEG event TSVs
- preprocessed EEG event TSVs
- stage-1 exclusion-union TSVs
- stage-2 EEG parcel NPY exports, including `*_PC1_gnorm.npy` and `*_time_sec.npy`
- stage-3 BOLD parcel NPY exports

## What `intermediate` means here

`intermediate` is the manuscript’s middle-ground TR-retention setting for EEG–BOLD alignment. Earlier development also explored `lenient` and `strict` variants, but those are not part of the final public manuscript pipeline. In practical terms, `intermediate` keeps TRs using the project’s compromise EEG coverage rules: not as permissive as `lenient`, and not as conservative as `strict`. The final downstream dataset used in this repo is the `intermediate`, `nolags`, `minlen15` branch.

## Manual Or Hybrid Dependency

This notebook is scripted, but it depends on earlier manual or hybrid work already being complete.

If you need the upstream manual context, see:
- `docs/manual_steps.md`
- especially the Brainstorm exclusion-marking and EEG source-workflow sections

## Important Trigger Dependency

This notebook explicitly depends on:
- recurring `R128` trigger events in both the raw and preprocessed EEG event TSVs
- a raw `S1` event used as the absolute anchor
- label columns that may store event codes in `trial_type`, `value`, or `type`
- time columns such as `onset`, `start`, `start_sec`, `time`, `latency_sec`, or `latency`

The current matching logic is intentionally preserved. It was built to tolerate event-label formats like `R128` and `Stimulus/R128` without silently changing the scientific alignment rule.

## Outputs Written

- per-run TR edges, coverage arrays, exclusion summaries, and keep masks
- per-run `bold_pc1.npy`
- per-run `eeg_power_tr.npy`
- per-run `eeg_power_tr_lags.npy`
- per-run `keep_center_*` masks and min-length summaries
- `qc/align_trmask_lags_summary.csv`
- `qc/run_input_audit.csv`
- `qc/alignment_parameters_used.json`


In [ ]:
# Step 0. Import Python modules and locate the Stage-4 helper file

import sys
from pathlib import Path

STAGE4_DIR = Path.cwd()
if not (STAGE4_DIR / "stage4_alignment_helpers.py").exists():
    candidate = Path.cwd() / "notebooks" / "4_alignment"
    if candidate.exists():
        STAGE4_DIR = candidate

if str(STAGE4_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE4_DIR.resolve()))

from stage4_alignment_helpers import run_alignment_batch


In [ ]:
# Step 1. User-editable input roots and preserved alignment settings
#
# Replace the <SET_...> placeholders with paths on your machine.
# Keep the filename patterns unchanged, because the preserved run-discovery
# logic depends on them.

RAW_EVENTS_DIR = Path("<SET_RAW_EVENTS_DIR>")
PREPROC_EVENTS_DIR = Path("<SET_PREPROC_EVENTS_DIR>")
EXCL_UNION_DIR = Path("<SET_EXCL_UNION_DIR>")
EEG_PARCEL_NPY_DIR = Path("<SET_EEG_PARCEL_NPY_DIR>")
BOLD_PARCEL_NPY_DIR = Path("<SET_BOLD_PARCEL_NPY_DIR>")
ALIGNMENT_OUTPUT_DIR = Path("<SET_ALIGNMENT_OUTPUT_DIR>")

TR_SEC = 2.1
EEG_FS_HZ = 250.0
LAGS_TR = (-1, 0, 1)
LAGS_TR_NO = (0,)

BASE_COVERAGE_THR = 0.70
HYBRID_MIN_COVERAGE = 0.50
HYBRID_MIN_GOOD_BLOCK_FRAC = 0.50
EEG_MIN_SAMPLES_FRAC = 0.65
EEG_MIN_SAMPLES_MIN = 50

DT_TOL_SEC = 0.15
OFFSET_JUMP_THR = 0.20

# Preserved implementation note:
# The original notebook exposed OFFSET_JUMP_THR here, but the active segment-boundary
# split inside the matching helper still uses a hard-coded 0.5-second threshold.
# This cleaned public notebook preserves that behavior and does not silently harmonize it.

REPORT_R128_TR_OFFSET = True
APPLY_R128_TR_OFFSET = False

MAKE_PLOTS = True
PLOT_RUNS = {"sub-16_ses-01"}
EXAMPLE_PARCEL_INDEX = 0


In [ ]:
# Step 2. Validate the configured roots and print a short run summary

def assert_configured_path(path_value, label):
    path_str = str(path_value)
    if "<SET_" in path_str:
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")

for label, path_value in [
    ("RAW_EVENTS_DIR", RAW_EVENTS_DIR),
    ("PREPROC_EVENTS_DIR", PREPROC_EVENTS_DIR),
    ("EXCL_UNION_DIR", EXCL_UNION_DIR),
    ("EEG_PARCEL_NPY_DIR", EEG_PARCEL_NPY_DIR),
    ("BOLD_PARCEL_NPY_DIR", BOLD_PARCEL_NPY_DIR),
    ("ALIGNMENT_OUTPUT_DIR", ALIGNMENT_OUTPUT_DIR),
]:
    assert_configured_path(path_value, label)

print("Stage 4 / Step 40: align EEG to BOLD TRs and build keep masks")
print(f"  Raw EEG events:        {RAW_EVENTS_DIR}")
print(f"  Preproc EEG events:    {PREPROC_EVENTS_DIR}")
print(f"  Exclusion unions:      {EXCL_UNION_DIR}")
print(f"  EEG parcel NPY root:   {EEG_PARCEL_NPY_DIR}")
print(f"  BOLD parcel NPY root:  {BOLD_PARCEL_NPY_DIR}")
print(f"  Output root:           {ALIGNMENT_OUTPUT_DIR}")
print(f"  TR:                    {TR_SEC}")
print(f"  EEG fs:                {EEG_FS_HZ}")
print(f"  Final no-lag branch:   lags={LAGS_TR_NO}")
print(f"  Optional lag branch:   lags={LAGS_TR}")
print(f"  Base threshold:        {BASE_COVERAGE_THR}")
print(f"  Hybrid threshold:      {HYBRID_MIN_COVERAGE}")
print(f"  Sample gate frac:      {EEG_MIN_SAMPLES_FRAC}")
print(f"  Trigger tolerance:     {DT_TOL_SEC}")
print("  Trigger requirement:   recurring R128 in raw + preproc, raw S1 anchor")


In [ ]:
# Step 3. Run the preserved alignment helper

result = run_alignment_batch(
    raw_events_dir=RAW_EVENTS_DIR,
    preproc_events_dir=PREPROC_EVENTS_DIR,
    excl_union_dir=EXCL_UNION_DIR,
    eeg_parcel_npy_dir=EEG_PARCEL_NPY_DIR,
    bold_parcel_npy_dir=BOLD_PARCEL_NPY_DIR,
    alignment_output_dir=ALIGNMENT_OUTPUT_DIR,
    tr_sec=TR_SEC,
    eeg_fs_hz=EEG_FS_HZ,
    lags_tr=LAGS_TR,
    lags_tr_no=LAGS_TR_NO,
    base_coverage_thr=BASE_COVERAGE_THR,
    hybrid_min_coverage=HYBRID_MIN_COVERAGE,
    hybrid_min_good_block_frac=HYBRID_MIN_GOOD_BLOCK_FRAC,
    eeg_min_samples_frac=EEG_MIN_SAMPLES_FRAC,
    eeg_min_samples_min=EEG_MIN_SAMPLES_MIN,
    dt_tol_sec=DT_TOL_SEC,
    offset_jump_thr=OFFSET_JUMP_THR,
    report_r128_tr_offset=REPORT_R128_TR_OFFSET,
    apply_r128_tr_offset=APPLY_R128_TR_OFFSET,
    make_plots=MAKE_PLOTS,
    plot_runs=PLOT_RUNS,
    example_parcel_index=EXAMPLE_PARCEL_INDEX,
)

print("Saved run-input audit:", result["audit_csv"])
print("Saved alignment summary:", result["qc_csv"])
print("Saved parameter record:", result["alignment_parameters_json"])
print("Ready runs:", len(result["ready_runs"]))

result["audit_df"].head(20)


In [ ]:
# Step 4. Review the missing-run audit and run-level alignment summary

print("\nRuns missing one or more required inputs:")
display(result["audit_df"].loc[~result["audit_df"]["ready"]].head(20))

print("\nRun-level alignment summary:")
result["qc_df"].head(20)
